# v0.2.8 verification run 4: external forest-validation provenance + parity (nothing else)

Runs ONLY: (1) the external forest-validation audit (`FCF_GV_FCF_GV_EMU_Ganjam_1`, `forest_loss_rate` only, ~3-4 min), regenerated into a **fresh
folder** so no leftover file can be zipped; (2) the parity harness (optional, ~90 min under restricted quota: it is byte-identical code to run 3,
see the diff printed in the last cell); (3) a provenance gate over the outputs. It does **not** rerun Deccan or the aquatic zone, adds no indicator and
changes no methodology.

What run 3 showed: the Ganjam_1 audit in the zip was the **run-2 file left on the Colab disk** (timestamp 08:13, no provenance block, commit 41496ed);
run 3 did not regenerate it. The validation zone uses the same `run_smoke_test` -> `write_audit` path as every other zone. The gate now rejects any
output with no provenance block, another commit, a dirty tree, a wrong contract version, or one written before this process imported the code.

**Restart the runtime first (Runtime -> Restart session), then run the cells in order.**

In [ ]:
# 1. Clone / update the v0.2.8 BRANCH (main is untouched). Safe to re-run.
import os, glob
REPO_DIR = '/content/reference-benchmarking'
BRANCH = 'v0.2.8-contract'
if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH https://github.com/G-auravSingh/reference-benchmarking.git $REPO_DIR
else:
    !git -C $REPO_DIR fetch origin $BRANCH
    !git -C $REPO_DIR checkout $BRANCH
    !git -C $REPO_DIR pull --ff-only origin $BRANCH
PKG_DIR = sorted(glob.glob(f'{REPO_DIR}/darukaa_reference_v*'))[-1]
!git -C $REPO_DIR log --oneline -1
%cd $PKG_DIR
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q pyproj shapely

In [ ]:
# 2. Earth Engine (same project as the main notebook)
import ee
ee.Authenticate()
ee.Initialize(project='gaurav-singh-007')
print('Earth Engine ready')

In [ ]:
# 3. Config: identical to the main notebook's Tata Motors config
from darukaa_reference.config import Config
from darukaa_reference.indicators import create_default_registry
config = Config(
    gee_project='gaurav-singh-007', output_dir='outputs', archetype='industrial',
    realm='terrestrial', assessment_mode='baseline',
    hmi_hard_ceiling=0.05, use_variance_stability_floor=True, reference_stratification='ecoregion_landcover',
)
registry = create_default_registry()
print('riparian ring width (m):', config.riparian_ring_width_m, '| early period:', config.net_change_early_years)

In [ ]:
# 3b. PROVENANCE GATE: what code did this process actually load? Stops if the checkout changed after import (stale kernel) or the tree is dirty.
from darukaa_reference import provenance as PV
prov = PV.run_provenance(config)
print(PV.brief(prov))
print('loaded commit:', prov['git_commit'], '| branch:', prov['branch'], '| on disk now:', prov['checkout_now'])
!git -C $REPO_DIR log --oneline -1
assert not prov['warnings'], 'PROVENANCE WARNING: ' + '; '.join(prov['warnings'])

In [ ]:
# 4. Locate the two zones from the Tata manifests
import sys, os
sys.path.insert(0, os.getcwd())
from run_project_from_manifest import find_manifest_by_project_name, load_manifest, resolve_tile_paths
REPO_ROOT = os.path.dirname(os.getcwd())
PATHS = {}
for proj in ('TataMotors_Pimpri', 'TataMotors_Pimpri_Aquatic'):
    mp = find_manifest_by_project_name(REPO_ROOT, proj)
    man = load_manifest(mp)
    PATHS.update(dict(zip(man['tile_labels'], resolve_tile_paths(man, mp))))
TERRESTRIAL_LABEL, AQUATIC_LABEL = 'EMU_Deccan_forest', 'Lake_Suman'
print(TERRESTRIAL_LABEL, '->', PATHS[TERRESTRIAL_LABEL]); print(AQUATIC_LABEL, '->', PATHS[AQUATIC_LABEL])

In [ ]:
# 5. Fresh output folder (never reuse outputs/smoke: it holds run-2 / run-3 files)
import os, shutil, dataclasses, json
from darukaa_reference import assess as A, provenance as PV
OUT = 'outputs/verify_run4'
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
RUN_PARITY = True          # False = skip the ~90 min parity rerun (its code is byte-identical to run 3; see the diff in the last cell)
print('output folder:', OUT, '| loaded commit:', PV.LOADED['commit'][:7])

## Step 1. External forest-validation audit (regenerated with real provenance)

In [ ]:
# 6. Forest validation zone, forest_loss_rate only. A methodological validation dataset: never in Tata scoring.
VALIDATION_LABEL = 'FCF_GV_FCF_GV_EMU_Ganjam_1'
cands = A.find_candidate_tiles(REPO_ROOT, top=12)
c = next((c for c in cands if c['label'] == VALIDATION_LABEL), None)
assert c, f'{VALIDATION_LABEL} not found among existing assets: ' + str([x["label"] for x in cands])
zoneV = dataclasses.replace(A.load_zone(config, registry, c['path'], c['label'], 'terrestrial'), validation_label=A.FOREST_VALIDATION_LABEL)
rowsV = A.run_smoke_test(config, registry, zoneV, out_dir=OUT, only=['forest_loss_rate'])
d = json.load(open(f'{OUT}/{zoneV.label}_audit.json'))
print({k: d['provenance'][k] for k in ('git_commit_short', 'branch', 'dirty', 'contract_version', 'package_version', 'code_version', 'warnings')})
print('forest reference diagnostics:', {k: v for k, v in json.loads(d['rows'][0]['provenance']).get('reference_construction', {}).items() if k in ('eligible_cells_total', 'n_requested_total', 'n_returned', 'shortfall', 'shortfall_note', 'leaf_summary')})

## Step 2. Parity (optional, same code as run 3)

In [ ]:
# 7. Parity harness -> OUT/parity.{md,json} (with provenance)
if RUN_PARITY:
    from darukaa_reference import parity
    zoneA = A.load_zone(config, registry, PATHS['EMU_Deccan_forest'], 'EMU_Deccan_forest', 'terrestrial')
    zoneB = A.load_zone(config, registry, PATHS['Lake_Suman'], 'Lake_Suman', 'aquatic')
    results = parity.run_parity(config, registry, zoneA, zoneB, out_dir=OUT, zone_forest=zoneV)
else:
    print('parity skipped by RUN_PARITY = False')

## Step 3. Provenance gate
Strict for the files this session wrote; the two main audits (Deccan, Lake_Suman) were produced by `9c2c4d3` in run 3 and are verified against that
commit, not against HEAD. The gate FAILS on a missing provenance block, another commit, dirty, wrong contract, or a leftover file.

In [ ]:
# 8. Gate + report. Stops (AssertionError) if any output has stale / missing / wrong provenance.
import subprocess
PRODUCING = '9c2c4d3'      # the commit that produced the run-3 Deccan / Lake_Suman audits (recorded in those files)
new = PV.verify_outputs(OUT)
old_dir = 'outputs/smoke'
old = PV.verify_outputs(old_dir, expected_commit=PRODUCING, check_session=False, only=['EMU_Deccan_forest_audit.json', 'Lake_Suman_audit.json']) if os.path.isdir(old_dir) else {'ok': None, 'files': [], 'issues': ['outputs/smoke not present (run-3 files not on this VM): verify them from your saved zip']}
for title, v in (('NEW (strict, this session)', new), (f'RUN-3 main audits vs {PRODUCING}', old)):
    print('\n==', title, '| ok =', v['ok'])
    for f in v['files']: print('  ', 'OK  ' if f['ok'] else 'FAIL', f['file'], f['commit'], f['problems'])
    for i in v['issues']: print('   ISSUE:', i)
print('\n== package diff between the run-3 commit and this checkout (scoring / reference / parity modules must be absent):')
print(subprocess.run(['git', '-C', REPO_DIR, 'diff', '--stat', PRODUCING, 'HEAD', '--', 'darukaa_reference_v0.2.7/darukaa_reference'], capture_output=True, text=True).stdout or '(no package changes)')
json.dump({'new': new, 'run3_main_audits': old, 'loaded': {k: PV.LOADED[k] for k in ('commit', 'branch', 'dirty', 'captured_utc')}}, open(f'{OUT}/verification_report.json', 'w'), indent=1, default=str)
assert new['ok'], 'NEW OUTPUTS FAILED THE PROVENANCE GATE'
assert old['ok'] in (True, None), 'RUN-3 MAIN AUDITS FAILED THE PROVENANCE CHECK'

In [ ]:
# 9. Zip the fresh folder only (leftovers in outputs/smoke are NOT included)
shutil.make_archive('/content/v028_verify_run4', 'zip', OUT)
from google.colab import files
files.download('/content/v028_verify_run4.zip')